In [ ]:
# ==========================================================
# 1) TRAINING — CUSTOM POOLING (streaming, no concat)
# ==========================================================
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


# ==========================================================
# Dataset: stream X_input_i.npy without concatenation
#   - loads files with mmap_mode='r'
#   - global index -> (file_id, local_id)
# ==========================================================
class NPYSplitDataset(Dataset):
    def __init__(self, folder, y_path, n_files=20, prefix="X_input_", mmap=True):
        self.folder = folder
        self.n_files = n_files
        self.paths = [os.path.join(folder, f"{prefix}{i}.npy") for i in range(n_files)]
        self.Y = np.load(y_path).astype(np.int64, copy=False)

        self._arrays = []
        self._lengths = []
        for p in self.paths:
            arr = np.load(p, mmap_mode="r" if mmap else None)
            self._arrays.append(arr)
            self._lengths.append(arr.shape[0])

        total = sum(self._lengths)
        assert total == len(self.Y), f"Total X ({total}) != len(Y) ({len(self.Y)})"

        self._offsets = np.cumsum([0] + self._lengths)  # [n_files+1]

    def __len__(self):
        return len(self.Y)

    def _locate(self, gidx):
        f = np.searchsorted(self._offsets, gidx, side="right") - 1
        local = gidx - self._offsets[f]
        return int(f), int(local)

    def __getitem__(self, idx):
        f, j = self._locate(idx)
        x = self._arrays[f][j]  # [F,W], float32
        y = self.Y[idx]
        x_t = torch.from_numpy(np.array(x, copy=False)).float().unsqueeze(0)  # [1,F,W]
        y_t = torch.tensor(int(y), dtype=torch.long)
        return x_t, y_t


# ==========================================================
# Custom Pooling (weighted average inside fixed groups)
# ==========================================================
class CustomAveragePooling(nn.Module):
    def __init__(self, out_segments, weights):
        super().__init__()
        self.out_segments = out_segments
        w = torch.tensor(weights, dtype=torch.float32)
        self.register_buffer("weights", w)

    def forward(self, x):
        B, C, H, W = x.shape

        seg_size = W // self.out_segments
        x = x[:, :, :, :seg_size * self.out_segments]

        x = x.view(B, C, H, self.out_segments, seg_size)

        w = self.weights[:seg_size]
        w = w / w.sum()
        w = w.view(1,1,1,1,seg_size)

        return (x * w).sum(dim=-1)


# ==========================================================
# Model — Custom pooling
# ==========================================================
import torch
import torch.nn as nn


class ModelCustomPooling(nn.Module):
    def __init__(self, n_features=82, d_hist=20, n_classes=15):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, (1, 3))
        self.pool1 = CustomAveragePooling(6, [0.6, 0.5, 0.4, 0.3, 0.2, 0.1])

        self.conv2 = nn.Conv2d(16, 16, (1, 2))
        self.pool2 = CustomAveragePooling(2, [0.6, 0.3])

        self.conv3 = nn.Conv2d(16, 16, (1, 2))

        self.flatten = nn.Flatten()

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_features, d_hist)
            dummy = torch.relu(self.conv1(dummy))
            dummy = self.pool1(dummy)
            dummy = torch.relu(self.conv2(dummy))
            dummy = self.pool2(dummy)
            dummy = torch.relu(self.conv3(dummy))
            flatten_size = dummy.numel()

        self.fc = nn.Sequential(
            nn.Linear(flatten_size, 32),
            nn.ReLU(),
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.relu(self.conv3(x))
        x = self.flatten(x)
        return self.fc(x)
    
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters     : {total:,}")
    print(f"Trainable parameters : {trainable:,}")

# ==========================================================
# Train — Custom pooling (NO reg)
# ==========================================================
def train_custom_pooling_streaming(
    train_folder="./X_input_split_train_smote",
    y_path="./X_input_split_train_smote/Y.npy",
    n_files=20,
    epochs=75,
    batch_size=256,
    lr=5e-5,
    n_features=82,
    d_hist=20,
    n_classes=15,
    num_workers=0,
    pin_memory=True,
    save_path="./cicids_models/model_custom_cicids.pth"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device utilisé :", device)

    ds = NPYSplitDataset(train_folder, y_path, n_files=n_files, prefix="X_input_", mmap=True)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    model = ModelCustomPooling(n_features=n_features, d_hist=d_hist, n_classes=n_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    count_parameters(model)
    total_start = time.time()

    for epoch in range(epochs):
        t0 = time.time()
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(loader, desc=f"CustomPooling | Epoch {epoch+1}/{epochs}", leave=False)
        for xb, yb in pbar:
            xb = xb.to(device, non_blocking=True)  # [B,1,F,W]
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            bs = yb.size(0)
            running_loss += loss.item() * bs
            pred = out.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += bs

            pbar.set_postfix(loss=float(loss.item()))

        dt = time.time() - t0
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"  Loss moyenne : {running_loss/total:.4f}")
        print(f"  Accuracy     : {100.0*correct/total:.2f}%")
        print(f"  Temps epoch  : {dt:.2f}s")
        print(f"  Samples/sec  : {total / max(1e-8, dt):.0f}")
        if torch.cuda.is_available():
            print(f"  GPU memory   : {torch.cuda.memory_allocated()/1024**2:.2f} MB")

    total_time = time.time() - total_start
    print(f"\nEntrainement terminé en {total_time/60:.2f} minutes")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print("✔ Saved:", save_path)
    return model


In [ ]:
# ==========================================================
# RUN (custom pooling)
# ==========================================================
print("--------------------Entrainement CustomPooling streaming--------------------")
_ = train_custom_pooling_streaming(
    train_folder="./X_input_split_train_smote",
    y_path="./X_input_split_train_smote/Y.npy",
    n_files=20,
    epochs=75,
    batch_size=256,
    lr=5e-5,
    n_features=82,
    d_hist=20,
    n_classes=15,
    num_workers=0,
    pin_memory=True,
    save_path="./cicids_models/model_custom_cicids.pth"
)